# Procesarea datelor

Material ONIA — Resurse educaționale. Exercițiile pot fi modificate și executate în JupyterLab.

## Un set de date este mai mult decât o matrice

În practică întâlnim fișiere CSV, JSON, tabele cu coloane numerice și categorice, valori lipsă, etichete incorecte și coloane de identificare. Primul pas este să înțelegem ce înseamnă fiecare rând și fiecare coloană. Un identificator nu este automat o caracteristică utilă: poate duce la memorarea exemplelor sau la scurgeri de informație. Verificăm numărul de observații, tipurile datelor, valori lipsă, duplicate, categorii și distribuția țintei. Abia apoi alegem o schemă de transformare și un model.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'vârstă': [15, 16, None, 15],
    'oraș': ['Iași', 'Cluj', 'Iași', 'Iași'],
    'scor': [82, 91, 77, 82]
})
print(df.info())
print(df.isna().sum())
print(df.select_dtypes(include='number').describe())

## Citirea și validarea unui CSV

Un CSV este un fișier text cu înregistrări, dar separatorul și regulile de citare pot varia. Un câmp poate conține și separatorul dacă este citat corespunzător. Pentru fișiere cu antet, pandas.read_csv este comod; pentru control elementar asupra fiecărui rând poate fi folosit modulul standard csv . După citire, verifică numele coloanelor, tipurile, câteva exemple și delimitatorul. Dacă separatorul este ; , folosește sep=';' . Nu presupune că toate numerele sunt recunoscute automat corect.

In [ ]:
import io
import pandas as pd
text = 'nume;scor\nAna;82\nDan;91\n'
df = pd.read_csv(io.StringIO(text), sep=';')
print(df.to_string(index=False))

## Valorile lipsă: ce înseamnă și cum le tratăm?

O valoare lipsă poate însemna „necolectat”, „nu se aplică” sau „necunoscut”. Nu o înlocuim mecanic înainte de a înțelege contextul. Soluții frecvente: eliminarea observațiilor/coloanelor când este justificat, imputare cu mediana sau media pentru valori numerice, moda pentru categorii și indicatori binari de lipsă. Imputarea învățată pe toate datele înainte de train/test split produce leakage. În Python, SimpleImputer trebuie potrivit doar pe train, ideal în Pipeline .

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer
X_train = np.array([[1., 7.], [2., np.nan], [3., 11.]])
X_test = np.array([[4., np.nan]])
imp = SimpleImputer(strategy='median')
print(imp.fit_transform(X_train))
print(imp.transform(X_test))

## Media versus mediana în prezența valorilor extreme

Media este influențată puternic de valori extreme: pentru 2, 3, 4, 5, 26, media este 8, mediana este 4. Dacă numărul 26 este o eroare de introducere, se poate investiga/corecta; dacă reprezintă un caz real rar, eliminarea automată poate fi dăunătoare. Mediana este robustă la valori extreme și e adesea o alegere potrivită pentru imputare; totuși, alegerea trebuie validată. În simulator poți schimba strategia și observa efectul asupra mediei și valorilor standardizate.

In [ ]:
from statistics import mean, median
v = [2, 3, 4, 5, 26]
print(mean(v), median(v))

## Duplicate și înregistrări incoerente

Două rânduri identice pot fi duplicări accidentale sau observații valide repetate. Nu șterge automat toate duplicatele: întreabă-te dacă fiecare rând reprezintă un eveniment diferit. Într-o competiție, două imagini identice sau aproape identice distribuite între train și validare pot produce o evaluare prea optimistă. Înainte de drop_duplicates() , stabilește care coloane definesc unic o observație. Verifică și reguli de domeniu: vârstă negativă, valori imposibile, clase în afara vocabularului.

In [ ]:
import pandas as pd
df = pd.DataFrame({'id': [1, 1, 2], 'valoare': [4, 4, 8]})
print(df.duplicated().tolist())
print(df.drop_duplicates().to_string(index=False))

## Detectarea valorilor extreme

Putem inspecta min/max, histograme, boxplot și intervale percentile. Metoda IQR folosește quartilele Q1, Q3 și intervalul intercuartilic IQR=Q3−Q1 ; observațiile în afara [Q1−1.5IQR, Q3+1.5IQR] sunt semnalate pentru investigație, nu automat greșite. În detectarea anomaliilor tocmai observațiile rare pot fi ținta modelului, deci eliminarea lor fără justificare poate distruge informație importantă.

In [ ]:
import numpy as np
v = np.array([3, 4, 4, 5, 6, 7, 29])
q1, q3 = np.percentile(v, [25, 75])
iqr = q3 - q1
marcate = (v < q1 - 1.5 * iqr) | (v > q3 + 1.5 * iqr)
print('Q1 Q3:', q1, q3)
print('Inspectăm:', v[marcate])

## StandardScaler, MinMaxScaler și RobustScaler

StandardScaler scade media train și împarte la abaterea standard train. MinMaxScaler rescalează după min/max train. RobustScaler folosește mediana și IQR și poate fi mai puțin afectat de observațiile extreme. Scalarea influențează mult metodele bazate pe distanță și coeficienți regularizați; arborii de decizie sunt de regulă puțin sensibili la scalarea monotonă a unei singure caracteristici numerice. Nu este o regulă universală pentru orice preprocesare sau orice model.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
X = np.array([[1.], [2.], [3.], [20.]])
for cls in [StandardScaler, MinMaxScaler, RobustScaler]:
    print(cls.__name__, cls().fit_transform(X).ravel().round(2))

## Pregătirea categoriilor și a textului

Categorii precum „Iași”, „Cluj” nu sunt numere; de aceea, modelul trebuie să primească o codificare adecvată. Pentru categorii nominale folosim adesea one-hot. Pentru text putem utiliza TfidfVectorizer ; acesta învață vocabularul din train, nu din datele finale ascunse. În datele de concurs, o categorie nevăzută în train nu trebuie să oprească programul. De aceea, una dintre opțiuni este OneHotEncoder(handle_unknown='ignore') .

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
texte_train = ['mere roșii', 'pere verzi', 'mere verzi']
texte_test = ['pere roșii']
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(texte_train)
X_test = vectorizer.transform(texte_test)
print(X_train.shape, X_test.shape)

## Pipeline complet pentru un set mixt

Împărțim întâi setul de date în train și validare. Construim o schemă pentru coloane numerice (imputare+scalare) și una pentru coloane categorice (imputare+one-hot). Combinăm transformările prin ColumnTransformer , apoi aplicăm un model de clasificare. Astfel, la fit , preprocesarea învață doar din train; la predict , transformările se aplică și datelor noi cu aceiași parametri. Acest model de organizare este deosebit de util în proiecte și competiții deoarece reduce erorile cauzate de procesarea inconsecventă.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
X = pd.DataFrame({'vârstă':[15,16,17,16,15,18],
                  'oraș':['Iași','Cluj','Iași','București','Cluj','Iași']})
y = [0, 0, 1, 0, 1, 1]
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), ['vârstă']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['oraș'])
])
model = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
model.fit(X, y)
print(model.predict(pd.DataFrame({'vârstă':[16], 'oraș':['Brașov']})))

## Flux reproductibil: fișier → model → predicții

Într-o soluție ONIA, creează un flux clar: citește fișierele, verifică schema, separă ținta, împarte datele pentru validare, antrenează transformările și modelul, calculează metrica potrivită, apoi aplică predict pe setul de concurs. Salvează doar rezultatele cerute, în ordinea rândurilor primite. Păstrează numele și ordinea coloanelor: o greșeală de ordine între train și test poate genera predicții invalide fără să apară neapărat o excepție.

In [ ]:
import pandas as pd
pred = pd.DataFrame({'id': [101, 102, 103],
                     'predictie': [0, 1, 0]})
pred.to_csv('submisie.csv', index=False, encoding='utf-8')
print(pred.to_string(index=False))

## Verificări finale înainte de evaluare

Verifică lungimea predicțiilor, numărul valorilor lipsă sau infinite, tipul ieșirii (probabilitate/clasă/valoare reală), coloanele cerute și ordinea observațiilor. Înregistrarea seed-ului și a versiunilor bibliotecilor îmbunătățește reproductibilitatea. Un rezultat bun trebuie să poată fi repetat; dacă nu poți reproduce scorul de validare cu aceeași configurație, investigați sursa aleatorului sau o posibilă scurgere de informație.

In [ ]:
import numpy as np
p = np.array([0.3, 0.75, 0.51])
assert p.shape == (3,)
assert np.isfinite(p).all()
assert np.all((p >= 0) & (p <= 1))
print('Verificări de bază trecute')